# Mushroom — uitgevoerde AutoML-vergelijking

De PyCaret 3.3.2-run is op 14 oktober 2026 werkelijk uitgevoerd in de oorspronkelijke
Python 3.11-omgeving. Dit notebook controleert de vastgelegde audit en presenteert
die bestaande resultaten apart. Het traint niet stilzwijgend een nieuwe selectie.
De bronuitvoer blijft in het oorspronkelijke projectnotebook staan. Stap 5 blijft
door Jorre af te ronden.

We controleren dat de bron en alle trainings- en testrijen dezelfde zijn als bij de handmatige modellen.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'MushroomFinal/mushroom_project_dataset.csv').exists())
sys.path.insert(0, str(ROOT))
from Final.tools.data import prepare, SOURCE_SHA256_LF
_, X, y, train, test, folds = prepare()
folder = ROOT/'MushroomFinal/comparison_correct'
audit = json.loads((folder/'pycaret_audit.json').read_text())
assert audit['train_source_rows'] == train.tolist() and audit['test_source_rows'] == test.tolist()
assert audit['dataset']['sha256_lf'] == SOURCE_SHA256_LF
display(pd.Series({k:v for k,v in audit.items() if k not in ['train_source_rows','test_source_rows']}))

version                                                                            3.3.2
train_rows                                                                          4100
test_rows                                                                            900
folds                                                                                  5
random_state                                                                          42
class_encoding                                                          {'e': 0, 'p': 1}
primary_metric                                                                  Macro F1
default_models_not_gridsearch_tuned                                                 True
dataset                                {'bytes': 155766, 'sha256': '1f215c4cf999b5df6...
all_model_rows                                                                        13
best_pipeline                          {'bytes': 18013866, 'sha256': '43035dd7e18c252...
macro_metric_validati

De tabel vergelijkt PyCaret-defaults; zij is niet dezelfde fit als de handmatig getunede kandidaten. De macro-F1-scorer en majority reference zijn afzonderlijk gecontroleerd.

In [2]:
results = pd.read_csv(folder/'pycaret_all.csv', index_col=0)
assert results.loc['dummy', 'Macro F1'] > .3
display(pd.read_csv(folder/'pycaret_three.csv', index_col=0))
display(results)
display(pd.read_csv(folder/'pycaret_test_metrics.csv'))

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Macro F1,TT (Sec)
rf,Random Forest Classifier,0.772683,0.809507,0.570240,0.770353,0.654987,0.491101,0.503717,0.742718,0.252
gbc,Gradient Boosting Classifier,0.733415,0.754756,0.399498,0.794778,0.531235,0.372416,0.415654,0.672468,0.336
lr,Logistic Regression,0.691707,0.681821,0.340243,0.687930,0.454306,0.272786,0.305873,0.619680,0.568


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,Macro F1,TT (Sec)
rf,Random Forest Classifier,0.772683,0.809507,0.570240,0.770353,0.654987,0.491101,0.503717,0.742718,0.264
et,Extra Trees Classifier,0.738780,0.777187,0.577314,0.684177,0.625170,0.427184,0.431497,0.712235,0.276
knn,K Neighbors Classifier,0.738049,0.753821,0.509696,0.716265,0.595113,0.409772,0.422923,0.700721,0.164
gbc,Gradient Boosting Classifier,0.733415,0.754756,0.399498,0.794778,0.531235,0.372416,0.415654,0.672468,0.350
dt,Decision Tree Classifier,0.685122,0.666559,0.590184,0.582667,0.586300,0.332168,0.332266,0.666042,0.092
ada,Ada Boost Classifier,0.694634,0.703307,0.380846,0.669802,0.484857,0.290971,0.314542,0.633879,0.172
lr,Logistic Regression,0.691707,0.681821,0.340243,0.687930,0.454306,0.272786,0.305873,0.619680,0.172
lda,Linear Discriminant Analysis,0.690000,0.678911,0.332509,0.687048,0.447319,0.267023,0.301132,0.615907,0.090
ridge,Ridge Classifier,0.690244,0.679329,0.326064,0.693337,0.442611,0.265429,0.301836,0.614019,0.076
svm,SVM - Linear Kernel,0.659268,0.657925,0.402705,0.587713,0.456145,0.228709,0.246241,0.601435,0.110


,rank,model,accuracy,balanced_accuracy,macro_f1,precision_p,recall_p,f1_p,roc_auc,average_precision,tn,fp,fn,tp
0,1,RandomForestClassifier,0.791111,0.751788,0.762992,0.807229,0.589443,0.681356,0.846159,0.792665,511,48,140,201
1,2,ExtraTreesClassifier,0.747778,0.722050,0.726136,0.686275,0.615836,0.649150,0.803196,0.739315,463,96,131,210
2,3,KNeighborsClassifier,0.758889,0.716127,0.725230,0.754098,0.539589,0.629060,0.784505,0.686941,499,60,157,184
3,4,GradientBoostingClassifier,0.747778,0.678592,0.683734,0.870130,0.392962,0.541414,0.808083,0.769251,539,20,207,134
4,5,DecisionTreeClassifier,0.725556,0.701302,0.703975,0.648734,0.601173,0.624049,0.701302,0.541113,448,111,136,205


Random Forest heeft de hoogste uitgevoerde PyCaret CV macro F1 (circa 0,7427). De handmatige RF300 scoort circa 0,7619 in dezelfde trainingsfolds. LightGBM crashte in de oorspronkelijke Windows-PyCaret-run en is transparant uitgesloten. Opnieuw uitvoeren kan met de volledige `MushroomFinal/requirements-comparison.txt` en `MushroomFinal/execute_pycaret_notebook.py`; doe dat in een aparte omgeving.